<a href="https://colab.research.google.com/github/srinithiarulprakash/smart-logistics-optimizer-/blob/main/Smart_Logistics_%26_Weather_Impact_E_Commerce_Optimizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests

# Example: Getting historical weather for London on 2011-12-01
# Latitude and Longitude for London
lat = 51.5074
lon = -0.1278
date = "2011-12-01"

url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={date}&end_date={date}&daily=precipitation_sum,temperature_2m_max"

response = requests.get(url)
data = response.json()

print(data)

{'latitude': 51.493847, 'longitude': -0.1630249, 'generationtime_ms': 0.1443624496459961, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 16.0, 'daily_units': {'time': 'iso8601', 'precipitation_sum': 'mm', 'temperature_2m_max': '°C'}, 'daily': {'time': ['2011-12-01'], 'precipitation_sum': [5.7], 'temperature_2m_max': [10.9]}}


In [11]:
import os
import pandas as pd

# Let's check where the file is located
if os.path.exists('/online_retail.csv'):
    file_path = '/online_retail.csv'
elif os.path.exists('/content/online_retail.csv'):
    file_path = '/content/online_retail.csv'
else:
    file_path = 'online_retail.csv'

print(f"Loading file from: {file_path}")
df = pd.read_csv(file_path, encoding='latin1')
print(f"Success! Total rows loaded: {len(df):,}")

Loading file from: /online_retail.csv
Success! Total rows loaded: 541,909


In [12]:
import pandas as pd
import requests


# 2. Basic cleaning (removing cancellations and missing customers)
df_clean = df[~df['InvoiceNo'].astype(str).str.startswith('C')].copy()
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
df_clean = df_clean.dropna(subset=['CustomerID'])
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

print(f"Base data loaded successfully. Total clean transactions: {len(df_clean):,}")

Base data loaded successfully. Total clean transactions: 397,884


In [13]:
import pandas as pd
import requests
from datetime import datetime
# 1. Map major destination countries to capital city coordinates (Latitude & Longitude
country_coords = {
    'United Kingdom':{'lat':51.5047, 'lon': -0.1278},
    'Germany': {'lat': 52.5200, 'lon': 13.4050},          # Berlin
    'France': {'lat': 48.8566, 'lon': 2.3522},           # Paris
    'EIRE': {'lat': 53.3498, 'lon': -6.2603},            # Dublin
    'Spain': {'lat': 40.4168, 'lon': -3.7038}
}
# 2. Extract date and filter for countries we have coordinates for
df_clean['DateOnly'] = df_clean['InvoiceDate'].dt.date
subset_df = df_clean[df_clean['Country'].isin(country_coords.keys())].copy()

# Get unique combinations of Country and Date to minimize API call
unique_pairs = subset_df[['Country','DateOnly']].drop_duplicates().reset_index(drop=True)
print(f"Ready to fetch weather data for {len(unique_pairs):,} unique country-date combinations")

Ready to fetch weather data for 941 unique country-date combinations


In [14]:
import time
import requests

weather_records = []

print("Fetching historical weather data from Open-Meteo API...")
for idx, row in unique_pairs.iterrows():
    country = row['Country']
    date_str = str(row['DateOnly'])
    lat = country_coords[country]['lat']
    lon = country_coords[country]['lon']

    # API Request URL for historical daily weather
    url = f"https://archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}&start_date={date_str}&end_date={date_str}&daily=precipitation_sum,temperature_2m_max"

    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            data = response.json()
            daily = data.get('daily', {})
            precipitation = daily.get('precipitation_sum', [0])[0]
            max_temp = daily.get('temperature_2m_max', [0])[0]

            weather_records.append({
                'Country': country,
                'DateOnly': row['DateOnly'],
                'Precipitation': precipitation if precipitation is not None else 0,
                'MaxTemp': max_temp if max_temp is not None else 0
            })
    except Exception as e:
        print(f"Error fetching for {country} on {date_str}: {e}")

    # Sleep briefly to be polite to the free API server
    time.sleep(0.02)

# Convert weather records to a DataFrame
weather_df = pd.DataFrame(weather_records)

# Merge weather data back into our transaction subset
enriched_df = pd.merge(subset_df, weather_df, on=['Country', 'DateOnly'], how='left')

print(f"Weather data merged successfully! Enriched dataset shape: {enriched_df.shape}")
print(enriched_df[['InvoiceNo', 'InvoiceDate', 'Country', 'Precipitation', 'MaxTemp']].head())

Fetching historical weather data from Open-Meteo API...
Error fetching for Germany on 2011-06-09: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=5)
Error fetching for Spain on 2011-06-09: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=5)
Error fetching for France on 2011-09-04: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=5)
Error fetching for EIRE on 2011-11-20: HTTPSConnectionPool(host='archive-api.open-meteo.com', port=443): Read timed out. (read timeout=5)
Weather data merged successfully! Enriched dataset shape: (381422, 11)
  InvoiceNo         InvoiceDate         Country  Precipitation  MaxTemp
0    536365 2010-12-01 08:26:00  United Kingdom            2.8      0.1
1    536365 2010-12-01 08:26:00  United Kingdom            2.8      0.1
2    536365 2010-12-01 08:26:00  United Kingdom            2.8      0.1
3    536365 2010-12-01 08:2

In [15]:
import numpy as np
# 1. Ensure TotalSales is calculated if not already
if 'TotalSales' not in enriched_df.columns:
  enriched_df['TotalSales'] = enriched_df['Quantity']*enriched_df['UnitPrice']
# 2. Create a 'Bad Weather Flag' (e.g., precipitation greater than 5mm indicates heavy rain/snow risk)
enriched_df['Bad_Weather_Flag'] = np.where(enriched_df['Precipitation']>5.0, 'High Risk','Normal')
# 3. Simulate a Logistics Delay Risk based on weather severity
# Heavy rain/precipitation increases the likelihood of a delivery delay
conditions = [
    (enriched_df['Precipitation']> 10.0),
    (enriched_df['Precipitation']> 2.0) & (enriched_df['Precipitation']<=10.0),
    (enriched_df['Precipitation']<=2.0)
]
choices = ['Delayed (>3Days)', 'Minor Delay (1-2 Days)', 'On-Time']
enriched_df['Fulfillment_Status'] = np.select(conditions, choices,default = 'On-Time')
# 4. Save the final enriched dataset to a new CSV file for Power BI / SQl
Output_file = 'enriched_retail_weather_data.csv'
enriched_df.to_csv(Output_file, index=False)
print(f"Feature engineering complete! Saved final datasetto '{Output_file}")
print(enriched_df[['InvoiceNo', 'Country', 'Precipitation', 'Bad_Weather_Flag', 'Fulfillment_Status']].head(10))

Feature engineering complete! Saved final datasetto 'enriched_retail_weather_data.csv
  InvoiceNo         Country  Precipitation Bad_Weather_Flag  \
0    536365  United Kingdom            2.8           Normal   
1    536365  United Kingdom            2.8           Normal   
2    536365  United Kingdom            2.8           Normal   
3    536365  United Kingdom            2.8           Normal   
4    536365  United Kingdom            2.8           Normal   
5    536365  United Kingdom            2.8           Normal   
6    536365  United Kingdom            2.8           Normal   
7    536366  United Kingdom            2.8           Normal   
8    536366  United Kingdom            2.8           Normal   
9    536367  United Kingdom            2.8           Normal   

       Fulfillment_Status  
0  Minor Delay (1-2 Days)  
1  Minor Delay (1-2 Days)  
2  Minor Delay (1-2 Days)  
3  Minor Delay (1-2 Days)  
4  Minor Delay (1-2 Days)  
5  Minor Delay (1-2 Days)  
6  Minor Delay (1-2 Days

In [16]:
from google.colab import files
files.download('enriched_retail_weather_data.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
print(enriched_df['Fulfillment_Status'].value_counts())

Fulfillment_Status
On-Time                   315158
Minor Delay (1-2 Days)     57351
Delayed (>3Days)            8913
Name: count, dtype: int64
